# Stage 6: Feature Engineering & Pre-Dispatch Modeling Pipeline

## 1. Project Context & Objectives
In Stage 5 (Exploratory Data Analysis), we established that customer dissatisfaction (1-star and 2-star reviews) is primarily driven by:
- Severe logistical bottlenecks (cross-state transit delays exceeding 3 days past SLA).
- Geographic friction (63.8% of volume moving inter-state, doubling delivery duration).
- High churn rates (96.88% one-time buyers).

The business goal of **Stage 6** is to construct a production-ready, tabular **Master ML Feature Matrix** to predict customer dissatisfaction **at the moment of order placement (pre-dispatch)**. This enables proactive intervention before a negative review is posted.

---

## 2. Key Pillars of Stage 6

### A. Binary Target Formulation (`is_dissatisfied`)
- **Dissatisfied (1):** Review score $\le 2$ (Severe churn and platform loss risk).
- **Satisfied (0):** Review score $\ge 3$ (Acceptable or positive customer experience).

### B. Strict Prevention of Data Leakage (Temporal Guardrails)
To build a realistic pre-dispatch early-warning system, features must only contain information available at checkout:
- **Excluded Features (Post-Fulfillment Leakage):** `order_delivered_customer_date`, `delay_days`, `review_score`, `review_comment_message`.
- **Allowed Features (Checkout State):** Delivery estimates (`order_estimated_delivery_date`), item prices, freight values, product volumetric size, seller-customer geographic distance, and temporal signals.

### C. Multi-Table Aggregation Architecture
The master table will be constructed at the unique `order_id` level by aggregating operational features from:
1. `df_orders`: Timestamps, baseline estimated transit windows.
2. `df_items` & `df_products`: Total items count, total value, volumetric weight, dominant product categories.
3. `df_customers` & `df_sellers`: Geographic distance flags (`is_cross_state`).
4. `df_payments`: Payment method complexity and installment behavior.
5. `df_reviews`: Isolated strictly to derive the ground-truth target variable (`is_dissatisfied`).

---

## 3. Workflow Execution Roadmap
- **Step 6.0:** Environment Setup & Interim Dataset Ingestion
- **Step 6.1:** Target Variable Creation (`is_dissatisfied`) & Class Balance Audit
- **Step 6.2:** Logistics & Geography Feature Extraction (`is_cross_state`, `estimated_transit_days`)
- **Step 6.3:** Order-Level Item & Product Dimensions Aggregation
- **Step 6.4:** Payment Behavior Aggregation
- **Step 6.5:** Temporal & Seasonality Feature Engineering
- **Step 6.6:** Master Dataset Consolidation & Final Leakage Verification

## Step 6.0: Environment Setup & Interim Dataset Ingestion
**Purpose:** Load essential data processing libraries (pandas, numpy) and ingest all 7 clean intermediate datasets from the `../data/interim/` directory into memory.

In [1]:
import pandas as pd
import numpy as np

# Interim data path
data_dir = "../data/interim"

# 8 datasets load karein
df_orders = pd.read_csv(f"{data_dir}/clean_orders.csv")
df_reviews = pd.read_csv(f"{data_dir}/clean_reviews.csv")
df_items = pd.read_csv(f"{data_dir}/clean_items.csv")
df_customers = pd.read_csv(f"{data_dir}/clean_customers.csv")
df_sellers = pd.read_csv(f"{data_dir}/clean_sellers.csv")
df_payments = pd.read_csv(f"{data_dir}/clean_payments.csv")
df_products = pd.read_csv(f"{data_dir}/clean_products.csv")


print("All 7 interim datasets successfully loaded into memory!")

All 7 interim datasets successfully loaded into memory!


## Step 6.1: Base Dataset Filtering & Target Variable Creation
**Purpose:** Filter for completed orders (`order_status == 'delivered'`), merge with customer review scores, and formulate our binary classification target `is_dissatisfied`:
- `is_dissatisfied = 1` for `review_score` <= 2 (High Risk / Dissatisfied)
- `is_dissatisfied = 0` for `review_score` >= 3 (Satisfied)

In [2]:
# 1. Filter only delivered orders to establish the evaluation cohort
ml_base = df_orders[df_orders["order_status"] == "delivered"].copy()

# 2. Join customer review scores using primary key 'order_id'
ml_base = ml_base.merge(df_reviews[["order_id", "review_score"]], on="order_id", how="inner")

# 3. Deduplicate orders to retain a single ground-truth review per order
ml_base = ml_base.drop_duplicates(subset=["order_id"]).reset_index(drop=True)

# 4. Define the binary target variable (1 for dissatisfied, 0 for satisfied)
ml_base["is_dissatisfied"] = (ml_base["review_score"] <= 2).astype(int)

# Inspect initial records
ml_base[["order_id", "review_score", "is_dissatisfied"]].head()

,order_id,review_score,is_dissatisfied
0,e481f51cbdc54678b7cc49136f2d6af7,4,0
1,53cdb2fc8bc7dce0b6741e2150273451,4,0
2,47770eb9100c2d0c44946d9cf07ec65d,5,0
3,949d5b44dbf5de918fe9c16f97b45f8a,5,0
4,ad21c59c0840e6cb83a9ceb5573f8159,5,0


## Step 6.1.1: Target Class Distribution Audit
**Purpose:** Quantify the class imbalance between satisfied and dissatisfied customers to establish the baseline metric for predictive modeling.

In [3]:
# Aggregate count of target classes
target_counts = ml_base["is_dissatisfied"].value_counts().reset_index()
target_counts.columns = ["is_dissatisfied", "order_count"]

# Compute percentage breakdown across target categories
total_orders = target_counts["order_count"].sum()
target_counts["percentage"] = (target_counts["order_count"] / total_orders * 100).round(2)

# Map descriptive classification labels
target_counts["label"] = target_counts["is_dissatisfied"].map({
    0: "Satisfied (Scores 3-5)",
    1: "Dissatisfied (Scores 1-2)"
})

display(target_counts[["label", "order_count", "percentage"]])

,label,order_count,percentage
0,Satisfied (Scores 3-5),83565,87.2
1,Dissatisfied (Scores 1-2),12267,12.8


## Step 6.2: Geographic & Logistics Feature Engineering (Pre-Dispatch Signals)
**Purpose:** Engineer core operational features available strictly at the time of purchase:
- `is_cross_state`: Binary indicator flag checking if seller state differs from customer state.
- `estimated_transit_days`: Promised delivery turnaround window promised to the customer at checkout.

In [4]:
# 1. Extract customer state by linking ml_base with df_customers
ml_base = ml_base.merge(
    df_customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

# 2. Extract seller state from items table (group by order_id to get the first seller state)
order_seller_state = df_items.merge(
    df_sellers[["seller_id", "seller_state"]],
    on="seller_id",
    how="left"
).groupby("order_id")["seller_state"].first().reset_index()

# 3. Merge seller state into master base
ml_base = ml_base.merge(order_seller_state, on="order_id", how="left")

# 4. Create binary cross-state flag (1 if different states, 0 if same state)
ml_base["is_cross_state"] = (ml_base["customer_state"] != ml_base["seller_state"]).astype(int)

# Inspect customer-seller geographic alignment
ml_base[["order_id", "customer_state", "seller_state", "is_cross_state"]].head()

,order_id,customer_state,seller_state,is_cross_state
0,e481f51cbdc54678b7cc49136f2d6af7,SP,SP,0
1,53cdb2fc8bc7dce0b6741e2150273451,BA,SP,1
2,47770eb9100c2d0c44946d9cf07ec65d,GO,SP,1
3,949d5b44dbf5de918fe9c16f97b45f8a,RN,MG,1
4,ad21c59c0840e6cb83a9ceb5573f8159,SP,SP,0


## Step 6.2.1: Estimated Delivery Window Extraction
**Purpose:** Calculate the promised turnaround days between purchase timestamp and estimated delivery date without leaking post-fulfillment actuals.

In [5]:
# Convert order timestamps to datetime format
ml_base["order_purchase_timestamp"] = pd.to_datetime(ml_base["order_purchase_timestamp"])
ml_base["order_estimated_delivery_date"] = pd.to_datetime(ml_base["order_estimated_delivery_date"])

# Compute planned transit duration in days
ml_base["estimated_transit_days"] = (
    ml_base["order_estimated_delivery_date"] - ml_base["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 3600)

# Round transit days to two decimal places
ml_base["estimated_transit_days"] = ml_base["estimated_transit_days"].round(2)

# Inspect calculated pre-dispatch features
ml_base[["order_id", "estimated_transit_days", "is_cross_state", "is_dissatisfied"]].head()

,order_id,estimated_transit_days,is_cross_state,is_dissatisfied
0,e481f51cbdc54678b7cc49136f2d6af7,15.54,0,0
1,53cdb2fc8bc7dce0b6741e2150273451,19.14,1,0
2,47770eb9100c2d0c44946d9cf07ec65d,26.64,1,0
3,949d5b44dbf5de918fe9c16f97b45f8a,26.19,1,0
4,ad21c59c0840e6cb83a9ceb5573f8159,12.11,0,0


## Step 6.3: Order-Level Item & Product Dimensions Aggregation
**Purpose:** Aggregate items and product specifications at the unique `order_id` level. Extract total price, freight, freight-to-price ratio, parcel weight, and volumetric capacity before dispatch.

In [8]:
# 1. Merge items with product dimensions to get physical specifications
items_with_products = df_items.merge(
    df_products[[
        "product_id", 
        "product_weight_g", 
        "product_length_cm", 
        "product_height_cm", 
        "product_width_cm"
    ]],
    on="product_id",
    how="left"
)

# 2. Calculate volumetric size in cubic centimeters (cm³) per item
items_with_products["product_volume_cm3"] = (
    items_with_products["product_length_cm"] *
    items_with_products["product_height_cm"] *
    items_with_products["product_width_cm"]
)

# 3. Aggregate metrics at the unique order_id level
order_item_agg = items_with_products.groupby("order_id").agg(
    total_items_count=("order_item_id", "count"),
    total_order_price=("price", "sum"),
    total_freight_value=("freight_value", "sum"),
    total_weight_g=("product_weight_g", "sum"),
    total_volume_cm3=("product_volume_cm3", "sum")
).reset_index()

# 4. Compute freight-to-price burden ratio (avoid zero division with small epsilon)
order_item_agg["freight_to_price_ratio"] = (
    order_item_agg["total_freight_value"] / (order_item_agg["total_order_price"] + 1e-5)
).round(4)

# 5. Merge aggregated item attributes into our master base dataframe
ml_base = ml_base.merge(order_item_agg, on="order_id", how="left")

# Inspect initial aggregated feature records
ml_base[[
    "order_id", 
    "total_items_count", 
    "total_order_price", 
    "total_freight_value", 
    "freight_to_price_ratio", 
    "total_weight_g"
]].head()

,order_id,total_items_count,total_order_price,total_freight_value,freight_to_price_ratio,total_weight_g
0,e481f51cbdc54678b7cc49136f2d6af7,1,29.99,8.72,0.2908,500.0
1,53cdb2fc8bc7dce0b6741e2150273451,1,118.70,22.76,0.1917,400.0
2,47770eb9100c2d0c44946d9cf07ec65d,1,159.90,19.22,0.1202,420.0
3,949d5b44dbf5de918fe9c16f97b45f8a,1,45.00,27.20,0.6044,450.0
4,ad21c59c0840e6cb83a9ceb5573f8159,1,19.90,8.72,0.4382,250.0


## Step 6.4: Order-Level Payment Behavior Aggregation
**Purpose:** Aggregate transaction methods, split payment complexity, and installment depth at the `order_id` level prior to dispatch.

In [9]:
# Aggregate payment attributes at the unique order_id level
order_payment_agg = df_payments.groupby("order_id").agg(
    payment_installments_max=("payment_installments", "max"),
    payment_sequential_count=("payment_sequential", "max"),
    preferred_payment_type=("payment_type", "first")
).reset_index()

# Merge payment attributes into our master dataframe
ml_base = ml_base.merge(order_payment_agg, on="order_id", how="left")

# Inspect merged payment behavioral features
ml_base[[
    "order_id", 
    "payment_installments_max", 
    "payment_sequential_count", 
    "preferred_payment_type"
]].head()

,order_id,payment_installments_max,payment_sequential_count,preferred_payment_type
0,e481f51cbdc54678b7cc49136f2d6af7,1.0,3.0,credit_card
1,53cdb2fc8bc7dce0b6741e2150273451,1.0,1.0,boleto
2,47770eb9100c2d0c44946d9cf07ec65d,3.0,1.0,credit_card
3,949d5b44dbf5de918fe9c16f97b45f8a,1.0,1.0,credit_card
4,ad21c59c0840e6cb83a9ceb5573f8159,1.0,1.0,credit_card


## Step 6.5: Final Pipeline Assembly & Leakage Guardrails
**Purpose:** Extract temporal purchase signals (day of week, hour, month), prune dataset truncation artifacts (Sept-Oct 2018), drop post-fulfillment leakage columns, and export the clean feature matrix to `../data/processed/`.

In [10]:
# 1. Extract temporal signals from purchase timestamp
ml_base["purchase_dayofweek"] = ml_base["order_purchase_timestamp"].dt.dayofweek
ml_base["purchase_hour"] = ml_base["order_purchase_timestamp"].dt.hour
ml_base["purchase_month"] = ml_base["order_purchase_timestamp"].dt.month

# 2. Prune data cutoff noise discovered in Stage 5 (September & October 2018 truncation)
ml_final = ml_base[ml_base["order_purchase_timestamp"] < "2018-09-01"].copy()

# 3. Drop all post-fulfillment leakage columns and raw string IDs
leakage_and_id_cols = [
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "review_score",          # Ground truth target source, must not leak into X
    "customer_state",
    "seller_state"
]

ml_final = ml_final.drop(columns=[col for col in leakage_and_id_cols if col in ml_final.columns])

# 4. Fill missing numerical values with median
num_cols = ml_final.select_dtypes(include=["float64", "int64"]).columns
ml_final[num_cols] = ml_final[num_cols].fillna(ml_final[num_cols].median())

# 5. Export final ready-to-train dataset to processed directory
ml_final.to_csv("../data/processed/final_ml_dataset.csv", index=False)

print(f"Master ML Dataset Shape: {ml_final.shape}")
print("Final ML dataset successfully exported to '../data/processed/final_ml_dataset.csv'!")
display(ml_final.head(3))

Master ML Dataset Shape: (95832, 29)
Final ML dataset successfully exported to '../data/processed/final_ml_dataset.csv'!


,order_id,order_purchase_year_month,is_dissatisfied,is_cross_state,estimated_transit_days,total_items_count_x,total_order_price_x,total_freight_value_x,total_weight_g_x,total_volume_cm3_x,...,total_freight_value,total_weight_g,total_volume_cm3,freight_to_price_ratio,payment_installments_max,payment_sequential_count,preferred_payment_type,purchase_dayofweek,purchase_hour,purchase_month
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10,0,0,15.54,1,29.99,8.72,500.0,1976.0,...,8.72,500.0,1976.0,0.2908,1.0,3.0,credit_card,0,10,10
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07,0,1,19.14,1,118.70,22.76,400.0,4693.0,...,22.76,400.0,4693.0,0.1917,1.0,1.0,boleto,1,20,7
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08,0,1,26.64,1,159.90,19.22,420.0,9576.0,...,19.22,420.0,9576.0,0.1202,3.0,1.0,credit_card,2,8,8


In [11]:
# 1. Drop duplicate columns ending with '_x' or '_y' if created during multiple merges
cols_to_drop = [c for c in ml_final.columns if c.endswith("_x") or c.endswith("_y")]
ml_final = ml_final.drop(columns=cols_to_drop)

# 2. Also drop 'order_purchase_year_month' if present (string date column not usable directly in ML)
if "order_purchase_year_month" in ml_final.columns:
    ml_final = ml_final.drop(columns=["order_purchase_year_month"])

# 3. Save the clean feature matrix to processed folder
ml_final.to_csv("../data/processed/final_ml_dataset.csv", index=False)

print(f"Cleaned Dataset Shape: {ml_final.shape}")
print("Columns list:", ml_final.columns.tolist())

Cleaned Dataset Shape: (95832, 16)
Columns list: ['order_id', 'is_dissatisfied', 'is_cross_state', 'estimated_transit_days', 'total_items_count', 'total_order_price', 'total_freight_value', 'total_weight_g', 'total_volume_cm3', 'freight_to_price_ratio', 'payment_installments_max', 'payment_sequential_count', 'preferred_payment_type', 'purchase_dayofweek', 'purchase_hour', 'purchase_month']
